# 구글 코랩으로 진행됨

## SPARK DATAFRAME API

### 1. spark 환경 설정 및 세션 연결

In [1]:
!apt-get install openjdk-8-jdk-headless #jdk install
!wget -q http://archive.apache.org/dist/spark/spark-3.0.0/spark-3.0.0-bin-hadoop3.2.tgz #spark file
!tar -xf spark-3.0.0-bin-hadoop3.2.tgz
!pip install findspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-8-jdk-headless is already the newest version (8u422-b05-1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.


- api-get: 시스템에서 사용 가능한 패키지에 대한 설치, 패키지 검색, 업데이트 및 기타 여러 작업 수행
- wget: 웹 상의 파일을 다운로드받을 때 사용
- tar: 여러 개의 파일을 하나의 파일로 묶거나 풀 때 사용
- pip: 파이썬에서 외부 라이브러리(패키지)를 설치, 업그레이드, 제거 검색 등의 작업 수행

In [ ]:
import os
import findspark

# 환경변수에 path 지정
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.0.0-bin-hadoop3.2"

# spark의 경우 잘 찾지 못하는 경우가 있어 findspark를 이용
findspark.init()

In [3]:
from pyspark.sql import SparkSession, Row
from pyspark.sql import types as T
from pyspark.sql import window as W
from pyspark.sql import functions as F

In [4]:
spark = SparkSession.builder.master("local").appName("Colab").getOrCreate()

In [5]:
# spark 객체 설정 확인
spark.sparkContext.getConf().getAll()

[('spark.master', 'local'),
 ('spark.app.name', 'Colab'),
 ('spark.rdd.compress', 'True'),
 ('spark.serializer.objectStreamReset', '100'),
 ('spark.driver.port', '43339'),
 ('spark.app.id', 'local-1730248835305'),
 ('spark.submit.pyFiles', ''),
 ('spark.executor.id', 'driver'),
 ('spark.submit.deployMode', 'client'),
 ('spark.driver.host', '3e8c587ab4b6'),
 ('spark.ui.showConsoleProgress', 'true')]

In [6]:
os.getcwd()

'/content'

### 2. 데이터 읽기

In [7]:
df = spark.read.option("header", "true").csv("./sample_data/test1.csv")
# option("header", "true"): 첫 번째 줄을 헤더로 인식하여 컬럼 이름으로

In [ ]:
df.show()
# 괄호 안에 숫자 입력 시 그만큼의 row만 조회
# vertical: 세로 정렬(true  지정 시 하나의 행마다 세로로 나옴)
# truncate: 정렬 (True 지정 시 오른쪽 정렬). 숫자 지정 시 앞에서부터 숫자만큼의 길이로 잘라서 반환
# name: Krish일 때 truncate=3을 하면 Kri로 출력됨

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [10]:
# check shema
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- Experience: string (nullable = true)
 |-- Salary: string (nullable = true)



In [11]:
# check data type
df.dtypes

[('Name', 'string'),
 ('age', 'string'),
 ('Experience', 'string'),
 ('Salary', 'string')]

In [16]:
# 일부 row만 보기
df.limit(2).show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
+---------+---+----------+------+



In [17]:
# check columns
df.columns

['Name', 'age', 'Experience', 'Salary']

In [18]:
# check row count
df.count()

6

In [19]:
# view row type
df.collect()

[Row(Name='Krish', age='31', Experience='10', Salary='30000'),
 Row(Name='Sudhanshu', age='30', Experience='8', Salary='25000'),
 Row(Name='Sunny', age='29', Experience='4', Salary='20000'),
 Row(Name='Paul', age='24', Experience='3', Salary='20000'),
 Row(Name='Harsha', age='21', Experience='1', Salary='15000'),
 Row(Name='Shubham', age='23', Experience='2', Salary='18000')]

In [20]:
# 첫 행 조회
df.first()

Row(Name='Krish', age='31', Experience='10', Salary='30000')

In [22]:
first_row = df.first()
first_row.Name, first_row.age, first_row.Experience
# 변수 지정 후 .column_name 하면 그 행의 열에 해당하는 값 출력

('Krish', '31', '10')

### 3. 컬럼 다루기 및 DataFrame 생성

In [26]:
# 컬럼 선택
df.select(["Name", "age"]).show()

+---------+---+
|     Name|age|
+---------+---+
|    Krish| 31|
|Sudhanshu| 30|
|    Sunny| 29|
|     Paul| 24|
|   Harsha| 21|
|  Shubham| 23|
+---------+---+



In [28]:
# 컬럼 삭제
df_drop = df.drop("Experience")
df_drop.show()

+---------+---+------+
|     Name|age|Salary|
+---------+---+------+
|    Krish| 31| 30000|
|Sudhanshu| 30| 25000|
|    Sunny| 29| 20000|
|     Paul| 24| 20000|
|   Harsha| 21| 15000|
|  Shubham| 23| 18000|
+---------+---+------+



In [ ]:
# 컬럼 추가
df_add = df.withColumn("age1", F.col("age"))

df_add.show()

+---------+---+----------+------+----+
|     Name|age|Experience|Salary|age1|
+---------+---+----------+------+----+
|    Krish| 31|        10| 30000|  31|
|Sudhanshu| 30|         8| 25000|  30|
|    Sunny| 29|         4| 20000|  29|
|     Paul| 24|         3| 20000|  24|
|   Harsha| 21|         1| 15000|  21|
|  Shubham| 23|         2| 18000|  23|
+---------+---+----------+------+----+



In [35]:
# 컬럼 추가 -expr
# 기존의 모든 컬럼과 함께 새로운 컬럼 생성
df_add.select("*", F.expr("age * 2 as age2")).show()

+---------+---+----------+------+----+----+
|     Name|age|Experience|Salary|age1|age2|
+---------+---+----------+------+----+----+
|    Krish| 31|        10| 30000|  31|62.0|
|Sudhanshu| 30|         8| 25000|  30|60.0|
|    Sunny| 29|         4| 20000|  29|58.0|
|     Paul| 24|         3| 20000|  24|48.0|
|   Harsha| 21|         1| 15000|  21|42.0|
|  Shubham| 23|         2| 18000|  23|46.0|
+---------+---+----------+------+----+----+



In [36]:
# 컬럼 추가 - selectExpr
# 컬럼을 명시적으로 선택하여 보고자 하는 컬럼만 볼 수 있음 + 새로운 컬럼 추가
df_add.selectExpr("Name", "age", "Experience", "age * 2 as age2").show()

+---------+---+----------+----+
|     Name|age|Experience|age2|
+---------+---+----------+----+
|    Krish| 31|        10|62.0|
|Sudhanshu| 30|         8|60.0|
|    Sunny| 29|         4|58.0|
|     Paul| 24|         3|48.0|
|   Harsha| 21|         1|42.0|
|  Shubham| 23|         2|46.0|
+---------+---+----------+----+



In [37]:
df_fixed = df

In [39]:
# 컬럼명 변경
# df.withColumnRenamed("기존 컬럼명", "변경할 컬럼명")
df_fixed.withColumnRenamed("age", "new_age").show()

+---------+-------+----------+------+
|     Name|new_age|Experience|Salary|
+---------+-------+----------+------+
|    Krish|     31|        10| 30000|
|Sudhanshu|     30|         8| 25000|
|    Sunny|     29|         4| 20000|
|     Paul|     24|         3| 20000|
|   Harsha|     21|         1| 15000|
|  Shubham|     23|         2| 18000|
+---------+-------+----------+------+



In [42]:
# 컬럼 타입 변경
df_fixed.withColumn("age", F.col("age").cast(T.IntegerType())).printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: string (nullable = true)
 |-- Salary: string (nullable = true)



In [ ]:
# DataFrame 생성
schema = T.StructType(
    [
        T.StructField("aaa", T.StringType(), True),
        T.StructField("bbb", T.StringType(), True),
        T.StructField("ccc", T.StringType(), True),
    ]
)

data_1 = [
    Row("apple", "banana", "tomato"),
    Row("apple1", "banana1", "tomato1"),
    Row("apple2", "banana2", "tomato2"),
]

data_2 = [
    ("apple", "banana", "tomato"),
    ("apple3", "banana3", "tomato3"),
    ("apple4", "banana4", "tomato4"),
]

df_new = spark.createDataFrame(data_2, schema)
df_new.show()

+------+-------+-------+
|   aaa|    bbb|    ccc|
+------+-------+-------+
| apple| banana| tomato|
|apple3|banana3|tomato3|
|apple4|banana4|tomato4|
+------+-------+-------+



### 4. DataFrame 쓰기 및 옵션

#### 1) Read - 기본 옵션
  - PERMISSIVE(default): 타입 이상 있으면 null 처리 후 read
  - DROPMALFORMED: 타입 이상이 있는 row drop
  - FAILFAST: 타입 이상이 있으면 fail

In [ ]:
_schema = "Name string, age int, Experience string, salary int"

name = (
    spark.read.schema(_schema)
    .option("mode", "PERMISSIVE")
    .csv("./sample_data/test1.csv", header=True, inferSchema=True)
)
name.show()

+---------+---+----------+------+
|     Name|age|Experience|salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [ ]:
# options
_option = {
    "header": "true",
    "InferSchema": "true",
    "mode": "PERMISSIVE",
    "columnnameOfCorruptRecord": "bad_record",
}

_schema = "Name string, age int, Experience string, salary int"

name = spark.read.schema(_schema).options(**_option).csv("./sample_data/test1.csv")
name.show()

+---------+---+----------+------+
|     Name|age|Experience|salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



#### 2) WRITE - 기본 옵션
- error(default): 파일이 있으면 실행 실패
- ignore: 파일이 있으면 작업 시도 안함
- overwrite: 기존 경로 덮어쓰기
- append: 기존 경로에 추가

- df.write.mode().option().csv()
- 하나의 파일로 저장: coalesce(1)

#### 3) WRITE - partition 옵션
- df.write.partitionBy("파티션할 컬럼명").mode("overwrite").csv("저장할 경로")

### 5. 결측치 다루기

#### 1) drop
- dropna()
- na.drop()

In [54]:
df = spark.read.csv("./sample_data/test2.csv", header=True)

df.show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [55]:
df_na1 = df.dropna()
df_na1.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [56]:
df_na1 = df.na.drop()
df_na1.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [57]:
# thresh: 정상값이 thresh개보다 적게 있는 row drop
df_na2 = df.dropna(thresh=2)
df_na2.show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
+---------+----+----------+------+



In [59]:
# any: null값이 하나라도 있으면 제거
# all: 모든 row가 null인 row 제거
df_na3 = df.dropna(how="all")
df_na3.show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [60]:
# 특정 컬럼만 적용
# subset에 지정한 컬럼들이 null인 경우 drop
df_na4 = df.dropna(how="all", subset=["age", "Experience"])
df_na4.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
|     null| 34|        10| 38000|
|     null| 36|      null|  null|
+---------+---+----------+------+



#### 2) fill
- fillna()
- na.fill()

In [62]:
# 컬럼 데이터 타입이 동일해야 가능
df_fill1 = df.fillna(value=1, subset=["age", "Experience", "Salary"])
df_fill1.show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



### 6. filter

In [63]:
item = spark.read.parquet("./sample_data/item_his.parquet")

item.show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|76257|190.0| 20230

In [ ]:
# 컬럼의 특정 값에 대한 행만 필터링
item.filter(F.col("codename") == "액세서리").show(10)

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|53687|175.0| 202305|20230508|액세서리|아바타파츠구분|   50|
|20163|161.0| 202304|20230408|액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|액세서리|아바타파츠구분|   50|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|67188|175.0| 202306|20230612|액세서리|아바타파츠구분|   50|
|11987|159.0| 202304|20230404|액세서리|아바타파츠구분|   50|
|48378|144.0| 202304|20230412|액세서리|아바타파츠구분|   50|
|32960|151.0| 202305|20230510|액세서리|아바타파츠구분|   25|
|61823|151.0| 202304|20230418|액세서리|아바타파츠구분|   25|
+-----+-----+-------+--------+--------+--------------+-----+
only showing top 10 rows



In [65]:
item.printSchema()

root
 |-- idx: string (nullable = true)
 |-- lv: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- codename: string (nullable = true)
 |-- mascodename: string (nullable = true)
 |-- price: string (nullable = true)



In [69]:
item = item.withColumn("price", F.col("price").cast(T.IntegerType()))
item.printSchema()

root
 |-- idx: string (nullable = true)
 |-- lv: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- codename: string (nullable = true)
 |-- mascodename: string (nullable = true)
 |-- price: integer (nullable = true)



In [71]:
item.filter(F.col("price") < 50).show(10)

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|32960|151.0| 202305|20230510|액세서리|아바타파츠구분|   25|
|61823|151.0| 202304|20230418|액세서리|아바타파츠구분|   25|
|77389|118.0| 202304|20230421|액세서리|아바타파츠구분|   25|
|13456|158.0| 202304|20230421|액세서리|아바타파츠구분|   25|
|74784|156.0| 202306|20230619|액세서리|아바타파츠구분|   25|
|86844|152.0| 202306|20230609|액세서리|아바타파츠구분|   25|
|52543|152.0| 202304|20230424|액세서리|아바타파츠구분|   25|
|43710|174.0| 202305|20230520|액세서리|아바타파츠구분|   25|
+-----+-----+-------+--------+--------+--------------+-----+
only showing top 10 rows



In [ ]:
# 다중 필터
item.filter(F.col("codename") == "액세서리").filter(F.col("price") < 30).show(10)

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|32960|151.0| 202305|20230510|액세서리|아바타파츠구분|   25|
|61823|151.0| 202304|20230418|액세서리|아바타파츠구분|   25|
|77389|118.0| 202304|20230421|액세서리|아바타파츠구분|   25|
|13456|158.0| 202304|20230421|액세서리|아바타파츠구분|   25|
|74784|156.0| 202306|20230619|액세서리|아바타파츠구분|   25|
|86844|152.0| 202306|20230609|액세서리|아바타파츠구분|   25|
|52543|152.0| 202304|20230424|액세서리|아바타파츠구분|   25|
|43710|174.0| 202305|20230520|액세서리|아바타파츠구분|   25|
+-----+-----+-------+--------+--------+--------------+-----+
only showing top 10 rows



In [80]:
# or 조건
item.filter((F.col("codename") == "액세서리") | (F.col("price") < 50)).show(10)

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|53687|175.0| 202305|20230508|액세서리|아바타파츠구분|   50|
|20163|161.0| 202304|20230408|액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|액세서리|아바타파츠구분|   50|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|67188|175.0| 202306|20230612|액세서리|아바타파츠구분|   50|
|11987|159.0| 202304|20230404|액세서리|아바타파츠구분|   50|
|48378|144.0| 202304|20230412|액세서리|아바타파츠구분|   50|
|32960|151.0| 202305|20230510|액세서리|아바타파츠구분|   25|
|61823|151.0| 202304|20230418|액세서리|아바타파츠구분|   25|
+-----+-----+-------+--------+--------+--------------+-----+
only showing top 10 rows



In [79]:
# 필터 반대 조건
item.filter(~(F.col("codename") == "액세서리")).show(10)

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|
+-----+-----+-------+--------+----------+--------------+-----+
only showing top 10 rows



In [81]:
# 중복 제거: distinct, dropDuplicates, drop_duplicates
item.distinct().show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|35264|174.0| 202305|20230520|      얼굴|아바타파츠구분|  150|
|84515| 83.0| 202304|20230425|      상의|아바타파츠구분|  200|
|94583| 46.0| 202306|20230611|      헤어|아바타파츠구분|  150|
|63082|162.0| 202306|20230611|      신발|아바타파츠구분|   60|
|89470|130.0| 202306|20230605|      얼굴|아바타파츠구분|  150|
| 4554|188.0| 202306|20230625|      얼굴|아바타파츠구분|  150|
|25030|182.0| 202306|20230605|      얼굴|아바타파츠구분|  150|
|89100|125.0| 202306|20230605|      헤어|아바타파츠구분|  150|
|  874|155.0| 202304|20230404|      헤어|아바타파츠구분|  150|
|43601|155.0| 202304|20230420|      신발|아바타파츠구분|   60|
|89811| 22.0| 202304|20230421|      상의|아바타파츠구분|  200|
|24195| 95.0| 202304|20230405|      신발|아바타파츠구분|   60|
|89530| 60.0| 202305|20230509|      헤어|아바타파츠구분|  150|
|17275|166.0| 202305|20230510|      헤어|아바타파츠구분|  150|
|30173|177.0| 202305|20230511|      얼굴|아바타파츠구분|  150|
|

In [82]:
item.dropDuplicates(subset=["codename", "price"]).show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|81788| 15.0| 202304|20230420|상태메시지|  기타파츠구분|  100|
|84801|112.0| 202306|20230627|      하의|아바타파츠구분|  250|
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|67188|175.0| 202306|20230612|      상의|아바타파츠구분|  200|
|76257|190.0| 202306|20230608|  액세서리|아바타파츠구분|   25|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|77041|169.0| 202306|20230628|      상의|아바타파츠구분|  150|
|67188|175.0| 202306|20230612|      신발|아바타파츠구분|   60|
|92027| null| 202305|20230505|    코스튬|아바타파츠구분|  300|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
+-----+-----+-------+--------+----------+--------------+-----+



In [84]:
# 컬럼에 특정 vlaue 포함 여부 확인
item.filter(F.col("codename").isin(["액세서리", "헤어"])).show(10)

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|53687|175.0| 202305|20230508|액세서리|아바타파츠구분|   50|
|20163|161.0| 202304|20230408|액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|    헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|    헤어|아바타파츠구분|  150|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|84801|112.0| 202306|20230627|    헤어|아바타파츠구분|  150|
+-----+-----+-------+--------+--------+--------------+-----+
only showing top 10 rows



In [87]:
# null 값 확인: isNull(), isNotNull()
item.filter(F.col("codename").isNotNull()).show()

+-----+-----+-------+--------+----------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|  codename|   mascodename|price|
+-----+-----+-------+--------+----------+--------------+-----+
|53687|175.0| 202305|20230508|  액세서리|아바타파츠구분|   50|
|53687|175.0| 202305|20230508|상태메시지|  기타파츠구분|   50|
|20163|161.0| 202304|20230408|  액세서리|아바타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|88572| 14.0| 202304|20230408|상태메시지|  기타파츠구분|   50|
|26112|130.0| 202304|20230405|  액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|상태메시지|  기타파츠구분|   50|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|      헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|37822| 39.0| 202306|20230608|상태메시지|  기타파츠구분|   50|
|76257|190.0| 20230

In [89]:
# 비슷한 데이터 형태 확인
item.filter(F.col("mascodename").like("아바타%")).show()

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|53687|175.0| 202305|20230508|액세서리|아바타파츠구분|   50|
|20163|161.0| 202304|20230408|액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|액세서리|아바타파츠구분|   50|
|26112|130.0| 202304|20230405|    헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|    헤어|아바타파츠구분|  150|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|76257|190.0| 202306|20230608|액세서리|아바타파츠구분|   25|
|84801|112.0| 202306|20230627|    헤어|아바타파츠구분|  150|
|84801|112.0| 202306|20230627|    하의|아바타파츠구분|  250|
|84801|112.0| 202306|20230627|    얼굴|아바타파츠구분|  150|
|67188|175.0| 202306|20230612|    얼굴|아바타파츠구분|  150|
|67188|175.0| 202306|20230612|    상의|아바타파츠구분|  

In [90]:
df = spark.read.csv("./sample_data/test2.csv", header=True)

df.show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [ ]:
# then 조건문
df.withColumn(
    "new_age", F.when(F.col("age").isNull(), "-").otherwise(F.col("Salary") * 2)
).show()

+---------+----+----------+------+-------+
|     Name| age|Experience|Salary|new_age|
+---------+----+----------+------+-------+
|    Krish|  31|        10| 30000|60000.0|
|Sudhanshu|  30|         8| 25000|50000.0|
|    Sunny|  29|         4| 20000|40000.0|
|     Paul|  24|         3| 20000|40000.0|
|   Harsha|  21|         1| 15000|30000.0|
|  Shubham|  23|         2| 18000|36000.0|
|   Mahesh|null|      null| 40000|      -|
|     null|  34|        10| 38000|76000.0|
|     null|  36|      null|  null|   null|
+---------+----+----------+------+-------+



In [93]:
# between: 그 사이에 존재하는 값만 조회
item.filter(F.col("price").between(100, 200)).show(10)

+-----+-----+-------+--------+--------+--------------+-----+
|  idx|   lv|proc_ym|proc_ymd|codename|   mascodename|price|
+-----+-----+-------+--------+--------+--------------+-----+
|26112|130.0| 202304|20230405|    헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    얼굴|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    신발|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    헤어|아바타파츠구분|  150|
|49541|170.0| 202306|20230630|    헤어|아바타파츠구분|  150|
|37822| 39.0| 202306|20230608|    헤어|아바타파츠구분|  150|
|84801|112.0| 202306|20230627|    헤어|아바타파츠구분|  150|
|84801|112.0| 202306|20230627|    얼굴|아바타파츠구분|  150|
|67188|175.0| 202306|20230612|    얼굴|아바타파츠구분|  150|
|67188|175.0| 202306|20230612|    상의|아바타파츠구분|  200|
+-----+-----+-------+--------+--------+--------------+-----+
only showing top 10 rows



### 7. Group by + aggregation

In [95]:
# group by, sum/count
item.groupBy(F.col("codename")).count().show()

+----------+-----+
|  codename|count|
+----------+-----+
|      신발|17245|
|      하의| 9198|
|  액세서리|11570|
|      얼굴|14075|
|      헤어|24011|
|    코스튬| 4697|
|상태메시지|17438|
|      상의|11311|
+----------+-----+



In [96]:
item.groupBy("codename").mean().show()

+----------+------------------+
|  codename|        avg(price)|
+----------+------------------+
|      신발| 74.39373731516382|
|      하의|             250.0|
|  액세서리| 44.81201382886776|
|      얼굴|             150.0|
|      헤어|             150.0|
|    코스튬|             300.0|
|상태메시지| 59.50510379630692|
|      상의|198.15224118115108|
+----------+------------------+



In [98]:
# agg: 여러 집계함수 동시에 적용 가능
item.groupBy("codename").agg(F.count("codename").alias("count")).show()

+----------+-----+
|  codename|count|
+----------+-----+
|      신발|17245|
|      하의| 9198|
|  액세서리|11570|
|      얼굴|14075|
|      헤어|24011|
|    코스튬| 4697|
|상태메시지|17438|
|      상의|11311|
+----------+-----+



In [99]:
item.groupBy("codename").agg(F.mean(F.col("price"))).alias("codename_mean").show()

+----------+------------------+
|  codename|        avg(price)|
+----------+------------------+
|      신발| 74.39373731516382|
|      하의|             250.0|
|  액세서리| 44.81201382886776|
|      얼굴|             150.0|
|      헤어|             150.0|
|    코스튬|             300.0|
|상태메시지| 59.50510379630692|
|      상의|198.15224118115108|
+----------+------------------+



In [ ]:
item.groupBy("codename").agg(
    F.count("codename").alias("count"),
    F.avg("price").alias("avg_price"),
    F.sum("price").alias("total_price"),
    F.max("price").alias("max_price"),
    F.min("price").alias("min_price"),
).show()

+----------+-----+------------------+-----------+---------+---------+
|  codename|count|         avg_price|total_price|max_price|min_price|
+----------+-----+------------------+-----------+---------+---------+
|      신발|17245| 74.39373731516382|    1282920|      150|       60|
|      하의| 9198|             250.0|    2299500|      250|      250|
|  액세서리|11570| 44.81201382886776|     518475|       50|       25|
|      얼굴|14075|             150.0|    2111250|      150|      150|
|      헤어|24011|             150.0|    3601650|      150|      150|
|    코스튬| 4697|             300.0|    1409100|      300|      300|
|상태메시지|17438| 59.50510379630692|    1037650|      100|       50|
|      상의|11311|198.15224118115108|    2241300|      200|      150|
+----------+-----+------------------+-----------+---------+---------+



In [ ]:
item.groupby("codename").agg(
    F.collect_list(F.col("price")).alias("codename_list"),
    F.collect_set(F.col("price")).alias("codename_set"),  # 중복값 제거
).show()

+----------+--------------------+------------+
|  codename|       codename_list|codename_set|
+----------+--------------------+------------+
|      신발|[150, 60, 60, 60,...|   [150, 60]|
|      하의|[250, 250, 250, 2...|       [250]|
|  액세서리|[50, 50, 50, 25, ...|    [50, 25]|
|      얼굴|[150, 150, 150, 1...|       [150]|
|      헤어|[150, 150, 150, 1...|       [150]|
|    코스튬|[300, 300, 300, 3...|       [300]|
|상태메시지|[50, 50, 50, 50, ...|   [100, 50]|
|      상의|[200, 200, 200, 2...|  [150, 200]|
+----------+--------------------+------------+



##### agg 집계함수
- sum: 총합
- countDist: 고유 개수
- count: 개수
- mean: 평균
- avg: 평균
- stddev: 표준편차
- min: 최소값
- max: 최대값
- round: 반올림(지정된 숫자를 자릿수로)
- collect_list: 컬럼값 리스트 형태로 수집
- collect_set: 컬럼 고유값 리스트 형태로 수집

### 8. Order by

In [109]:
df = df.withColumn("age", F.col("age").cast(T.IntegerType()))
df = df.withColumn("Experience", F.col("Experience").cast(T.IntegerType()))
df = df.withColumn("Salary", F.col("Salary").cast(T.IntegerType()))

df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [110]:
# 오름차순 정렬
df.orderBy("Salary").show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|     null|  36|      null|  null|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|Sudhanshu|  30|         8| 25000|
|    Krish|  31|        10| 30000|
|     null|  34|        10| 38000|
|   Mahesh|null|      null| 40000|
+---------+----+----------+------+



In [112]:
# 내림차순 정렬
df.orderBy(F.col("Salary").desc()).show()
# df.orderBy("age", ascending=False).show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|  Shubham|  23|         2| 18000|
|   Harsha|  21|         1| 15000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [113]:
# 여러 컬럼 지정 시 순차적으로 정렬(기본값은 오름차순)
# null값은 가장 작게 판단함
df.orderBy("age", "Experience").show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|   Mahesh|null|      null| 40000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|     Paul|  24|         3| 20000|
|    Sunny|  29|         4| 20000|
|Sudhanshu|  30|         8| 25000|
|    Krish|  31|        10| 30000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [127]:
# 나이는 오름차순, 급여는 내림차순으로 정렬
df.orderBy(F.col("age").desc(), F.col("Experience")).show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|     null|  36|      null|  null|
|     null|  34|        10| 38000|
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|  Shubham|  23|         2| 18000|
|   Harsha|  21|         1| 15000|
|   Mahesh|null|      null| 40000|
+---------+----+----------+------+



In [114]:
test = spark.read.csv("./sample_data/test1.csv", header=True)

test.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [119]:
test = test.withColumn("age", F.col("age").cast(T.IntegerType()))
test = test.withColumn("Experience", F.col("Experience").cast(T.IntegerType()))
test = test.withColumn("Salary", F.col("Salary").cast(T.IntegerType()))

In [121]:
test.orderBy("age", "Salary").show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
|     Paul| 24|         3| 20000|
|    Sunny| 29|         4| 20000|
|Sudhanshu| 30|         8| 25000|
|    Krish| 31|        10| 30000|
+---------+---+----------+------+



In [122]:
test.orderBy("Salary", "age").show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
|     Paul| 24|         3| 20000|
|    Sunny| 29|         4| 20000|
|Sudhanshu| 30|         8| 25000|
|    Krish| 31|        10| 30000|
+---------+---+----------+------+



### 9. Join & Union

#### 종류
- inner
-left
- right
- outer(full)
- left-semi: 왼쪽에 일치하는 행만 반환 (오른쪽 열 포함 X)
- left-anti: 왼쪽에 불일치하는 행만 반환 (오른쪽 열 포함 X)

In [132]:
df2 = spark.read.csv("./sample_data/test2.csv", header=True)

df2.show()
df2.printSchema()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+

root
 |-- Name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- Experience: string (nullable = true)
 |-- Salary: string (nullable = true)



In [130]:
df3 = spark.read.csv("./sample_data/test3.csv", header=True)

df3.show()
df3.printSchema()

+---------+------------+------+
|     Name| Departments|salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
|    Sunny|    Big Data|  2000|
+---------+------------+------+

root
 |-- Name: string (nullable = true)
 |-- Departments: string (nullable = true)
 |-- salary: string (nullable = true)



In [134]:
df2 = df2.withColumnRenamed("Name", "New_Name")

df2.show()

+---------+----+----------+------+
| New_Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [136]:
df2.join(df3, df2.New_Name == df3.Name, "outer").show()
# how: default는 inner. left, right, outer/full
# sql과 동일하게 왼쪽을 기준으로 오른쪽에 조인하는 테이블 적기

+---------+----+----------+------+---------+------------+------+
| New_Name| age|Experience|Salary|     Name| Departments|salary|
+---------+----+----------+------+---------+------------+------+
|     null|  34|        10| 38000|     null|        null|  null|
|     null|  36|      null|  null|     null|        null|  null|
|Sudhanshu|  30|         8| 25000|Sudhanshu|Data Science| 20000|
|Sudhanshu|  30|         8| 25000|Sudhanshu|         IOT| 10000|
|Sudhanshu|  30|         8| 25000|Sudhanshu|    Big Data|  5000|
|    Sunny|  29|         4| 20000|    Sunny|Data Science| 10000|
|    Sunny|  29|         4| 20000|    Sunny|    Big Data|  2000|
|    Krish|  31|        10| 30000|    Krish|Data Science| 10000|
|    Krish|  31|        10| 30000|    Krish|         IOT|  5000|
|    Krish|  31|        10| 30000|    Krish|    Big Data|  4000|
|   Harsha|  21|         1| 15000|     null|        null|  null|
|     Paul|  24|         3| 20000|     null|        null|  null|
|  Shubham|  23|         

In [139]:
df2.join(df3, df2.New_Name == df3.Name, "right").show()

+---------+----+----------+------+---------+------------+------+
| New_Name| age|Experience|Salary|     Name| Departments|salary|
+---------+----+----------+------+---------+------------+------+
|    Krish|  31|        10| 30000|    Krish|Data Science| 10000|
|    Krish|  31|        10| 30000|    Krish|         IOT|  5000|
|   Mahesh|null|      null| 40000|   Mahesh|    Big Data|  4000|
|    Krish|  31|        10| 30000|    Krish|    Big Data|  4000|
|   Mahesh|null|      null| 40000|   Mahesh|Data Science|  3000|
|Sudhanshu|  30|         8| 25000|Sudhanshu|Data Science| 20000|
|Sudhanshu|  30|         8| 25000|Sudhanshu|         IOT| 10000|
|Sudhanshu|  30|         8| 25000|Sudhanshu|    Big Data|  5000|
|    Sunny|  29|         4| 20000|    Sunny|Data Science| 10000|
|    Sunny|  29|         4| 20000|    Sunny|    Big Data|  2000|
+---------+----+----------+------+---------+------------+------+



In [142]:
# 여러 컬럼 join
df2.join(df3, (df2.New_Name == df3.Name) & (df2.Salary > df3.salary)).show()

+---------+----+----------+------+---------+------------+------+
| New_Name| age|Experience|Salary|     Name| Departments|salary|
+---------+----+----------+------+---------+------------+------+
|    Krish|  31|        10| 30000|    Krish|Data Science| 10000|
|   Mahesh|null|      null| 40000|   Mahesh|    Big Data|  4000|
|   Mahesh|null|      null| 40000|   Mahesh|Data Science|  3000|
|Sudhanshu|  30|         8| 25000|Sudhanshu|Data Science| 20000|
|Sudhanshu|  30|         8| 25000|Sudhanshu|         IOT| 10000|
|    Sunny|  29|         4| 20000|    Sunny|Data Science| 10000|
|    Sunny|  29|         4| 20000|    Sunny|    Big Data|  2000|
+---------+----+----------+------+---------+------------+------+



In [144]:
df1 = spark.read.csv("./sample_data/test1.csv", header=True)
df2 = spark.read.csv("./sample_data/test2.csv", header=True)

df1.show()
df2.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [145]:
# union: 테이블의 컬럼을 반드시 통일(순서 및 명칭)한 후 진행
# unionByName: 테이블 컬럼의 순서가 달라도 union 가능
df1.union(df2).show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|    Krish|  31|        10| 30000|
|Sudhanshu|  30|         8| 25000|
|    Sunny|  29|         4| 20000|
|     Paul|  24|         3| 20000|
|   Harsha|  21|         1| 15000|
|  Shubham|  23|         2| 18000|
|   Mahesh|null|      null| 40000|
|     null|  34|        10| 38000|
|     null|  36|      null|  null|
+---------+----+----------+------+



In [147]:
# 중복값 제거: 모든 컬럼의 데이터값이 동일한 행 중복 제거
df1.union(df2).distinct().show()

+---------+----+----------+------+
|     Name| age|Experience|Salary|
+---------+----+----------+------+
|  Shubham|  23|         2| 18000|
|   Harsha|  21|         1| 15000|
|Sudhanshu|  30|         8| 25000|
|     null|  34|        10| 38000|
|   Mahesh|null|      null| 40000|
|    Sunny|  29|         4| 20000|
|    Krish|  31|        10| 30000|
|     null|  36|      null|  null|
|     Paul|  24|         3| 20000|
+---------+----+----------+------+



### 10. Window

In [153]:
df3 = df3.withColumn("Salary", F.col("Salary").cast(T.IntegerType()))

df3.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Departments: string (nullable = true)
 |-- Salary: integer (nullable = true)



In [ ]:
from pyspark.sql import window as W

# window 변수
window_var = W.Window.partitionBy("Name").orderBy(
    "Salary"
)  # Name을 기준으로 급여 오름차순 정렬

# window 변수가 적용된 컬럼 생성
df3.withColumn(
    "window", F.row_number().over(window_var)
).show()  # vindow 변수의 정렬에 따라 Name그룹별 순서매김

+---------+------------+------+------+
|     Name| Departments|Salary|window|
+---------+------------+------+------+
|Sudhanshu|    Big Data|  5000|     1|
|Sudhanshu|         IOT| 10000|     2|
|Sudhanshu|Data Science| 20000|     3|
|    Sunny|    Big Data|  2000|     1|
|    Sunny|Data Science| 10000|     2|
|    Krish|    Big Data|  4000|     1|
|    Krish|         IOT|  5000|     2|
|    Krish|Data Science| 10000|     3|
|   Mahesh|Data Science|  3000|     1|
|   Mahesh|    Big Data|  4000|     2|
+---------+------------+------+------+



### 11. UDF(User Defined Function)
- DataFrame의 특정 컬럼을 사용자가 원하는 형태로 가공
- 데코레이터 사용 또는 함수 직접 명시

In [ ]:
def str_lower(var):
    return var.lower()


udf_lower = F.udf(str_lower, returnType=T.StringType())

df3.withColumn("lower_Departments", udf_lower(F.col("Departments"))).show()

+---------+------------+------+-----------------+
|     Name| Departments|Salary|lower_Departments|
+---------+------------+------+-----------------+
|    Krish|Data Science| 10000|     data science|
|    Krish|         IOT|  5000|              iot|
|   Mahesh|    Big Data|  4000|         big data|
|    Krish|    Big Data|  4000|         big data|
|   Mahesh|Data Science|  3000|     data science|
|Sudhanshu|Data Science| 20000|     data science|
|Sudhanshu|         IOT| 10000|              iot|
|Sudhanshu|    Big Data|  5000|         big data|
|    Sunny|Data Science| 10000|     data science|
|    Sunny|    Big Data|  2000|         big data|
+---------+------------+------+-----------------+



In [ ]:
@F.udf(returnType=T.StringType())
def str_lower(var):
    return var.lower()


df3.withColumn("lower_Departments", str_lower(F.col("Departments"))).show()

+---------+------------+------+-----------------+
|     Name| Departments|Salary|lower_Departments|
+---------+------------+------+-----------------+
|    Krish|Data Science| 10000|     data science|
|    Krish|         IOT|  5000|              iot|
|   Mahesh|    Big Data|  4000|         big data|
|    Krish|    Big Data|  4000|         big data|
|   Mahesh|Data Science|  3000|     data science|
|Sudhanshu|Data Science| 20000|     data science|
|Sudhanshu|         IOT| 10000|              iot|
|Sudhanshu|    Big Data|  5000|         big data|
|    Sunny|Data Science| 10000|     data science|
|    Sunny|    Big Data|  2000|         big data|
+---------+------------+------+-----------------+



### 12. 기타 유용한 함수들

In [165]:
# 컬럼명 일괄 변경

change_list = ["name", "depart", "salary"]
df3.toDF(*change_list).show()

+---------+------------+------+
|     name|      depart|salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
|    Sunny|    Big Data|  2000|
+---------+------------+------+



In [166]:
# F.lit(): 고정된 값 넣어주는 함수
df3.withColumn("lit_col", F.lit("lit_col")).show()

+---------+------------+------+-------+
|     Name| Departments|Salary|lit_col|
+---------+------------+------+-------+
|    Krish|Data Science| 10000|lit_col|
|    Krish|         IOT|  5000|lit_col|
|   Mahesh|    Big Data|  4000|lit_col|
|    Krish|    Big Data|  4000|lit_col|
|   Mahesh|Data Science|  3000|lit_col|
|Sudhanshu|Data Science| 20000|lit_col|
|Sudhanshu|         IOT| 10000|lit_col|
|Sudhanshu|    Big Data|  5000|lit_col|
|    Sunny|Data Science| 10000|lit_col|
|    Sunny|    Big Data|  2000|lit_col|
+---------+------------+------+-------+



In [171]:
# F.split(): 특정 문자를 기준으로 문자 split
df3.withColumn("split_col", F.split(F.col("Departments"), " ")[0]).show()

+---------+------------+------+---------+
|     Name| Departments|Salary|split_col|
+---------+------------+------+---------+
|    Krish|Data Science| 10000|     Data|
|    Krish|         IOT|  5000|      IOT|
|   Mahesh|    Big Data|  4000|      Big|
|    Krish|    Big Data|  4000|      Big|
|   Mahesh|Data Science|  3000|     Data|
|Sudhanshu|Data Science| 20000|     Data|
|Sudhanshu|         IOT| 10000|      IOT|
|Sudhanshu|    Big Data|  5000|      Big|
|    Sunny|Data Science| 10000|     Data|
|    Sunny|    Big Data|  2000|      Big|
+---------+------------+------+---------+



In [173]:
# pyspark DataFrame -> Pandas DF: toPandas()
# Pandas DataFrame -> pyspark DF: spark.createDataFrame
pdf = df.toPandas()
type(pdf)

pandas.core.frame.DataFrame

In [ ]:
# sdf = spark.createDataFrame(pdf)
# type(sdf)
# 버전 문제로 오류남

## Spark SQL

- 데이터를 처리하고 분석하기 위한 SLQ 인터페이스 및 엔진 + 데이터 및 메타데이터 관리
- SQL 쿼리로 Spark DataFrame 다룰 수 있도록 하는 기능

#### 1) 임시 뷰

In [208]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local").appName("Colab").getOrCreate()

In [209]:
spark.conf.get("spark.sql.catalogImplementation")
# spark sql이 기본적으로 사용하는 카탈로그 구현 방식 결정
# 'in-memory': 메모리에 카탈로그 정보 저장하여 관리. 임시 테이블, view 를 생성할 때 사용
# 'hive': 메타스토어를 통해 정보관리. Hive 테이블을 직접 다루거나 기본 Hive 메타데이터에 접근할 때 사용

'in-memory'

In [ ]:
name = spark.read.csv("./sample_data/test1.csv", header=True)

name.createOrReplaceTempView(
    "names"
)  # sql 쿼리에서 데이터프레임 사용할 수 있도록 임시 뷰 생성

In [182]:
spark.sql("SHOW DATABASES;").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [183]:
# 저장된 테이블 조회
spark.sql("SHOW TABLES FROM DEFAULT;").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|        |    names|       true|
+--------+---------+-----------+



In [185]:
# 테이블 내용 조회
spark.sql("SELECT * FROM NAMES;").show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [186]:
spark.read.table("names").show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local")
    .appName("Colab")
    .config("spark.sql.catalogImplementation", "in-memory")
    .getOrCreate()
)

In [211]:
spark.conf.get("spark.sql.catalogImplementation")

'in-memory'

In [192]:
# DataBase 생성 및 조회
spark.sql("CREATE DATABASE temp;")

spark.sql("SHOW DATABASES;").show()

+---------+
|namespace|
+---------+
|  default|
|     temp|
+---------+



In [195]:
# 테이블 생성
name.createOrReplaceTempView("names")

In [ ]:
spark.sql("SHOW TABLES FROM TEMP;").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|        |    names|       true|
+--------+---------+-----------+



- Member 데이터에서 회원상태(mbr_segment_mm)별 인원수 집계후 인원수 값을 내림차순 정렬하여 출력
  - 일별 회원의 마지막 상태 정의 방법: lv 컬럼의 값이 가장 높은 행

In [223]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import window as W

In [218]:
spark = SparkSession.builder.master("local").appName("Colab").getOrCreate()

In [221]:
mdf = spark.read.parquet("./sample_data/member.parquet")

mdf.show(5)

+------+---+--------+-------+
|   idx|sex|  status|  grade|
+------+---+--------+-------+
|   100| 남|유료회원|초1학년|
|  1000| 여|유료회원|초5학년|
| 10000| 여|유료회원|초6학년|
|100007| 남|유료회원|초4학년|
| 10001| 남|유료회원|초2학년|
+------+---+--------+-------+
only showing top 5 rows



In [222]:
mdf.printSchema()

root
 |-- idx: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- status: string (nullable = true)
 |-- grade: string (nullable = true)



In [226]:
_member_window = W.Window.partitionBy(F.col("idx")).orderBy(F.col("grade").desc())

In [ ]:
result = (
    mdf.withColumn("last_status", F.row_number().over(_member_window))
    .filter(F.col("last_status") == 1)
    .groupBy(F.col("status"))
    .count()
    .orderBy(F.col("count").desc())
)

In [228]:
result.show()

+--------+-----+
|  status|count|
+--------+-----+
|유료회원|66032|
|학습만료|   92|
|    신규|   75|
|  재구매|   70|
|    이월|   22|
|    복회|    7|
|    취소|    6|
+--------+-----+



#### point_his 데이터의 회원 ID(idx)별 포인트(point) 총합을 계산하고 포인트가 높은 순서로 정렬하여 데이터 출력 (소수점 제외)

In [230]:
point_df = spark.read.parquet("./sample_data/point_his.parquet")

point_df.show(5)
point_df.printSchema()

+-----+-------+--------+-----+
|  idx|proc_ym|proc_ymd|point|
+-----+-------+--------+-----+
|96465| 202306|20230624| 1000|
|96465| 202306|20230624|  500|
|87940| 202304|20230405| 2000|
|87940| 202304|20230405| 3500|
|87940| 202304|20230405| 4000|
+-----+-------+--------+-----+
only showing top 5 rows

root
 |-- idx: string (nullable = true)
 |-- proc_ym: string (nullable = true)
 |-- proc_ymd: string (nullable = true)
 |-- point: string (nullable = true)



In [ ]:
point_df.groupBy(F.col("idx")).agg(
    F.sum(F.col("point")).cast("int").alias("total_point")
).orderBy(F.col("total_point").desc()).show()

+-----+-----------+
|  idx|total_point|
+-----+-----------+
|92247|      82500|
|92845|      67500|
|93038|      63500|
|94232|      55000|
|91616|      55000|
|90260|      55000|
|88201|      55000|
|96687|      55000|
|93213|      55000|
|94418|      55000|
|95415|      55000|
|90442|      55000|
|91264|      55000|
|89167|      55000|
|95798|      55000|
|94243|      55000|
|84514|      55000|
|94373|      55000|
|90906|      55000|
|91828|      55000|
+-----+-----------+
only showing top 20 rows



- member 데이터의 회원grade가 0학년, 초4학년인 경우 상태를 '체험회원'으로 나머지는 '유료회원'형태로 집계

In [ ]:
res = mdf.withColumn(
    "status",
    F.when(
        (F.col("grade") == "0학년") | (F.col("grade") == "초4학년"), "체험회원"
    ).otherwise("유료회원"),
)

In [241]:
status_count = res.groupBy(F.col("status")).count()

In [242]:
status_count.show()

+--------+-----+
|  status|count|
+--------+-----+
|유료회원|50480|
|체험회원|15824|
+--------+-----+

